# TF-IDF terms

Compare discriminative terms for papers with and without explicit UK Biobank mentions in the full publication corpus.


In [ ]:
import sys
from pathlib import Path

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "utils").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils import shared_paths as P
from utils.data_analysis_00_dataset_analysis import (
    CATEGORY_PATTERNS,
    MODEL_NAMES,
    contains_pattern,
    load_publications,
    model_agreement_columns,
    normalise_bool,
    normalized_rows,
    output_dirs,
    sample_balanced,
    save_figure,
)

P.bootstrap()

from sklearn.feature_extraction.text import TfidfVectorizer


In [ ]:
df = load_publications(P.SHOWCASE_PLUS)
df.shape


In [ ]:
table_dir, figure_dir = output_dirs("00_dataset_analysis_05_tfidf_terms")
sample = sample_balanced(df, "explicit_ukb_mention", 20000, 42)
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=5, max_df=0.85, max_features=12000)
matrix = vectorizer.fit_transform(sample["analysis_text"].fillna(""))
terms = np.asarray(vectorizer.get_feature_names_out())
explicit = sample["explicit_ukb_mention"].to_numpy()
explicit_mean = np.asarray(matrix[explicit].mean(axis=0)).ravel()
other_mean = np.asarray(matrix[~explicit].mean(axis=0)).ravel()

term_summary = pd.DataFrame(
    {
        "term": terms,
        "mean_tfidf_explicit_ukb": explicit_mean,
        "mean_tfidf_no_explicit_ukb": other_mean,
        "difference_explicit_minus_other": explicit_mean - other_mean,
    }
).sort_values("difference_explicit_minus_other", ascending=False)
term_summary.to_csv(table_dir / "tfidf_discriminative_terms.csv", index=False)


In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(16, 9))
top_explicit = term_summary.head(25).iloc[::-1]
top_other = term_summary.tail(25).sort_values("difference_explicit_minus_other").iloc[::-1]
axes[0].barh(top_explicit["term"], top_explicit["difference_explicit_minus_other"])
axes[0].set(title="Associated with explicit UKB mentions", xlabel="Mean TF-IDF difference")
axes[1].barh(top_other["term"], -top_other["difference_explicit_minus_other"])
axes[1].set(title="Associated with no explicit UKB mention", xlabel="Mean TF-IDF difference")
for axis in axes:
    axis.grid(axis="x", alpha=0.25)
figure.tight_layout()
save_figure(figure, figure_dir / "tfidf_discriminative_terms.png")
term_summary.head(10)
